# Zero2FSD — Week 1 — Driving ML Gym

This notebook is the **lecture path** for Week 1.

- **Scaffold** (~60–70%): data loading, `DrivingClassifier`, cross-entropy training loop
- **Fill**: implement `focal_loss` and `minority_recall` in `modules/00_ml_gym/*.py`
- **From scratch**: implement `build_error_gallery` in `error_gallery.py`

Crops are **synthetic checked-in PNGs** under `data/m00_sample` (CC0 license), not random `torch.randn` tensors.

**Appendix 00b (optional, not this week's critical path):** `modules/00_nn_scratch` and `notebooks/00_neural_networks_and_autograd.ipynb` cover autograd from scratch when you want that depth.

## Session card

Zero2FSD · Week 1

**Today's win:** See pedestrian recall, not just accuracy, on the checked-in crops.

**Time:** 25 / 55 / 90 minutes

A day counts when you export an artifact or pass the tests for a fill you wrote. Opening the notebook does not. Set pause_week to true if you need a week off; the count stays where it is.

**Your stack so far**

- [ ] m00 — Driving ML Gym
- [ ] m01 — Cameras & IPM
- [ ] m02 — HydraNet
- [ ] m03 — BEV transform
- [ ] m04 — Occupancy
- [ ] m05 — Vector tracking
- [ ] m06 — Planning
- [ ] m07 — Control
- [ ] m08 — Capstone
- [ ] m09 — System architecture

XP is not awarded for opening this notebook.

In [ ]:
import sys
from pathlib import Path

repo = Path.cwd()
if not (repo / 'modules' / '00_ml_gym').exists():
    repo = repo.parent
sys.path.insert(0, str(repo / 'modules'))
from common.progress import format_stack

progress_path = repo / 'artifacts' / 'progress.json'
if progress_path.is_file():
    import json
    with progress_path.open() as f:
        progress = json.load(f)
    print(format_stack(progress))
else:
    print('No progress file yet. Opening this notebook awards 0 XP.')

**Governing principles (this week)**

1. **A model is a function from data to scores;** learning adjusts parameters so a chosen loss gets small on the training distribution.
2. **The loss defines what good means.** If the loss ignores rare classes, the model will too.
3. **You cannot improve what you do not inspect.** Headline accuracy hides confident mistakes.

**Why — these three, before the API.** The function, the loss, and the inspection are the whole week. Softmax, ReLU, convolution, and focal loss are shapes those three ideas take, not a separate subject. If you only remember function names, you can train a net and still certify it with accuracy on a road-heavy log. The fills exist so you apply the loss and the inspection yourself, in code, on these crops.

## 0. Cold open

### Theory

This week you train a function from a **64×64 driving crop** to **four class scores**: clear road, lead vehicle, pedestrian, lane marking (`CLASS_NAMES`). The scaffold `DrivingClassifier` already exists in `modules/00_ml_gym/model.py`. It will not be good yet — weights start random.

Inside each section we go **bottom-up**: short theory, tiny demo, check question, then the next idea.

Pedestrian is class index **2** in `CLASS_NAMES`.

**Why — start with a working crop task.** A full stack (cameras, bird's-eye view, planning) hides the one skill this week is about: fit a function, and notice when the number you track is the wrong one. A 64×64 crop is small enough to train here and concrete enough that a wrong class is a picture you can look at. You should see the failure before the vocabulary — the course direction is top-down, like FastAI, on this small task.

**Why — four classes, and why road dominates.** These labels are the confusions that matter inside one patch: empty drivable surface, a vehicle ahead, a person, and paint on the road. Real logs are mostly empty road, because that is what the cameras see most of the time. This sample keeps that skew on purpose. A balanced toy would hide the failure that matters: a model that looks accurate and still misses pedestrians.

**Why — the untrained net and the always-road rule.** Random weights have not seen a label, so argmax is noise. That is the baseline "nothing has been learned." The always-road rule is the other baseline: no network, just the majority class. It can score well on accuracy and still have pedestrian recall of zero. Later beats exist to beat both of those, not to decorate a training loop.

In [ ]:
import sys
from pathlib import Path
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

repo = Path.cwd()
if not (repo / 'modules' / '00_ml_gym').exists():
    repo = repo.parent
sys.path.insert(0, str(repo / 'modules'))
sys.path.insert(0, str(repo / 'modules' / '00_ml_gym'))
torch.manual_seed(0)

from config import TrainConfig
from dataset import CLASS_NAMES, get_dataloaders
from model import DrivingClassifier
from losses import cross_entropy_loss, focal_loss
from metrics import accuracy, per_class_recall, minority_recall
print('repo:', repo)

### Demo

In [ ]:
cfg = TrainConfig(data_dir=repo / 'data' / 'm00_sample')
train_loader, val_loader = get_dataloaders(cfg.data_dir, batch_size=16, seed=0)
images, labels = next(iter(train_loader))
print('train batches per epoch:', len(train_loader), 'train images:', len(train_loader.dataset))

In [ ]:
ds = train_loader.dataset
fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for ax, i in zip(axes.flat, range(8)):
    img, lab = ds[i]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(CLASS_NAMES[lab.item()], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample crops from PNG files')
plt.tight_layout()
plt.show()

In [ ]:
model = DrivingClassifier(num_classes=4)
model.eval()
v_images, v_labels = next(iter(val_loader))
with torch.no_grad():
    v_logits = model(v_images)
v_preds = v_logits.argmax(dim=-1)
v_acc = accuracy(v_preds, v_labels)
print('logits shape', v_logits.shape, '(B, 4)')
print('argmax preds', v_preds.tolist())
print('labels       ', v_labels.tolist())
print('val accuracy (untrained):', round(v_acc, 3))

In [ ]:
counts = train_loader.dataset.class_counts()
n = sum(counts.values())
count_road = counts[0]
always_road_acc = count_road / n
plt.bar(CLASS_NAMES, [counts[i] for i in range(4)])
plt.title('Train class counts')
plt.xticks(rotation=20)
plt.show()
print(f'Always-predict-clear_road accuracy: {always_road_acc:.3f} ({count_road}/{n})')
print('Pedestrian recall of that dumb rule: 0.0')

### Check

Which axis of a `(B, 3, 64, 64)` batch is **color**, and why does `Conv2d` care? The next cell checks pixel range and shape, then prints the axis convention.

In [ ]:
images, labels = next(iter(train_loader))
assert images.shape[1:] == (3, 64, 64), images.shape
assert float(images.min()) >= 0.0 and float(images.max()) <= 1.0
print('Channel axis is 1 because PyTorch image batches use (B, C, H, W).')

## 1. Images as numbers

### Theory

A PNG is a grid of pixels. Each pixel stores three numbers **R, G, B**, usually as integers 0–255 on disk. After load we **divide by 255** so each channel sits in **[0, 1]**. We do **not** subtract dataset mean/std in this module.

On disk and in PIL, layout is **(H, W, C)**: height, width, channels. Our `DrivingPatchDataset` permutes to **(C, H, W)** per image. A batch stacks images into **(B, C, H, W)** where **B** is batch size, **C** is channels (3 for RGB), **H** height, **W** width.

When you `imshow` a tensor, you often `permute(1, 2, 0)` back to HWC because Matplotlib expects channels last. Inside the network, keep channels first.

**Why — an image is an array, not a photo.** The net never sees "a pedestrian." It sees a table of numbers. If a crop looks like a person to you and like a similar blob of pixels to the arithmetic, the model can only learn the arithmetic. Plotting one channel as numbers is how you check that the file you think you loaded is the tensor you actually train on.

**Why — divide by 255, and stop there.** Raw values near 255 make the first weighted sums large, so a modest learning rate takes a huge step and the loss can diverge. Dividing by 255 is a fixed change of units, the same for every crop, so it does not teach the class. Subtracting a dataset mean and dividing by standard deviation is a later refinement. This week the requirement is a shared, bounded scale.

**Why — channels before height and width.** A convolution dots a kernel with a neighborhood and writes one number per output channel. PyTorch stores those channels on axis 1 so the kernel's input-channel count lines up with the tensor. Height-width-channel is how files and screens are stored. Leave a batch that way and Conv2d treats height as if it were the number of colors. The leading B is how many crops share one step, not a spatial axis. Separate R, G, and B matter on the road: a lane dash and a shirt can share brightness and differ in color. Collapsing to gray would throw that away before learning starts.

### Demo

In [ ]:
from PIL import Image
import numpy as np

row0 = train_loader.dataset.rows[0]
png_path = train_loader.dataset.data_dir / str(row0['filename'])
pil_img = Image.open(png_path).convert('RGB')
print('PIL size (W,H):', pil_img.size, 'mode:', pil_img.mode)
hwc = np.array(pil_img, dtype=np.float32) / 255.0
print('array shape (H,W,C):', hwc.shape)
print('Red channel 4x4 patch (rounded):\n', np.round(hwc[:4, :4, 0], 2))

In [ ]:
tensor_chw, lab = train_loader.dataset[0]
print('dataset tensor shape (C,H,W):', tuple(tensor_chw.shape))
hwc_from_tensor = tensor_chw.permute(1, 2, 0).numpy()
assert np.allclose(hwc_from_tensor, hwc, atol=1e-5)
print('permute(1,2,0) matches PIL array within tolerance')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(hwc)
axes[0].set_title('RGB image')
axes[0].axis('off')
im = axes[1].imshow(hwc[:, :, 0], cmap='viridis')
axes[1].set_title('Red channel as numbers')
plt.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

### Check

One crop is `(3, 64, 64)`. What shape is a stack of 16? The next cell builds the stack and asserts the batched shape.

In [ ]:
stacked = torch.stack([train_loader.dataset[i][0] for i in range(16)])
assert stacked.shape == (16, 3, 64, 64), stacked.shape
print('Sixteen images batched:', tuple(stacked.shape))

## 2. Classification as scores

### Theory

The model outputs **logits** \(z\): one raw score per class. They are **not** probabilities. **Softmax** turns one vector of logits into positive numbers that sum to 1:

$$p_k = \frac{\exp(z_k)}{\sum_j \exp(z_j)}$$

The predicted class is **argmax** of \(z\) (same as argmax of \(p\), because softmax preserves which coordinate is largest within the same vector).

If the true class is \(y\), **cross-entropy** charges the negative log probability of that class:

$$\mathrm{CE} = -\log p_y$$

**Hand example:** logits `[2.0, 1.0, 0.5, -1.0]`, true class **0**. Class 0 has the largest logit, so it gets the largest \(p_k\). CE is small when \(p_y\) is near 1 and grows without bound as \(p_y \to 0\) because \(\log p\) goes to \(-\infty\).

PyTorch `F.cross_entropy` uses a **numerically stable** `log_softmax` internally; the formula above is the definition.

Optional deepeners (after you can compute CE by hand): [StatQuest on cross-entropy](https://www.youtube.com/watch?v=6ArSys5qHAU), [3Blue1Brown neural networks](https://www.3blue1brown.com/lessons/neural-networks).

The scaffold wraps the same idea in `cross_entropy_loss(logits, targets)` for shape `(B, K)` and `(B,)`.

**Why — scores during learning, a class name only at the end.** At decision time, argmax can say "pedestrian." During learning, a hard label throws away how wrong you were. A true-class logit of 0.1 and a true-class logit of 8 can share an argmax and must not share an update. Scores keep a magnitude the loss can push.

**Why — softmax, not argmax alone.** Argmax is a decision. Softmax is the comparison that turns any real scores into positive weights summing to 1, so "negative log probability of the true class" is defined. Without the sum-to-one step, a model can raise every logit and look more confident without preferring the true class. Softmax forces a trade: probability spent on road is probability not spent on the pedestrian. That is the right constraint when the crop has one label.

**Why — cross-entropy, not a 0–1 grade.** A right-or-wrong loss is flat almost everywhere, so there is no gradient until the chosen class flips. Negative log grows without bound as the true-class probability goes to 0, and it is near 0 when that probability is near 1. A crop labeled pedestrian that the model calls road with probability 0.99 produces a large charge. A crop it already gets right produces a small one. The batch average is why one very wrong crop can still move the weights.

### Demo

In [ ]:
import torch.nn.functional as F

z = torch.tensor([2.0, 1.0, 0.5, -1.0])
y = torch.tensor(0)
p = F.softmax(z, dim=0)
manual_ce = -torch.log(p[y])
ce = F.cross_entropy(z.unsqueeze(0), y.unsqueeze(0))
print('softmax p', [round(x, 4) for x in p.tolist()])
print('manual CE', manual_ce.item(), 'F.cross_entropy', ce.item())
assert torch.allclose(manual_ce, ce, atol=1e-5)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(CLASS_NAMES, p.numpy())
ax.set_ylabel('probability')
ax.set_title('Softmax of example logits')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### Check

Same logits `[2, 1, 0.5, -1]`. If the true class is **3** instead of **0**, does CE go **up** or **down**? Why? The next cell prints both CE values and asserts the harder label costs more.

In [ ]:
y_wrong = torch.tensor(3)
ce_true = F.cross_entropy(z.unsqueeze(0), y.unsqueeze(0)).item()
ce_wrong = F.cross_entropy(z.unsqueeze(0), y_wrong.unsqueeze(0)).item()
print('CE true class 0:', ce_true)
print('CE true class 3 (weak logit):', ce_wrong)
assert ce_wrong > ce_true

## 3. Neuron and layer

### Theory

One **neuron** is an affine map plus a nonlinearity:

$$z = w \cdot x + b, \qquad y = \mathrm{ReLU}(z), \qquad \mathrm{ReLU}(z) = \max(0, z)$$

A **layer** is many neurons. With PyTorch `nn.Linear`, batch shape is **(B, F_in)** and weights have shape **(F_out, F_in)** so \(Y = \mathrm{ReLU}(X W^T + b)\).

This beat is not yet an image model; images come once you trust the train loop on a toy case.

**Why — a weighted sum plus a bias.** A score has to depend on the features, and different features count differently. In a crop, a vertical edge is not the same evidence as a bright patch. The weights are those importances. The bias is the score you would emit before seeing the features, a tilt. Together they are the smallest map that can rank one pattern above another.

**Why — ReLU, not another linear layer.** Two linear maps compose into one: \(W_2(W_1 x) = (W_2 W_1)x\). Depth without a bend does not add functions. You still have one flat cut through the input. ReLU zeros the negative side, so different regions of the input activate different weights. XOR on four corners is the small proof: one line cannot separate them, and a hidden layer with a bend can. "Pedestrian" in pixel space is a harder version of that fact. It is not one half-plane of the crop.

**Why — a 2-feature toy before the image net.** If the bend fails on four points, it will not succeed on 64×64×3. The toy keeps the failure on one plot. Batch norm inside DrivingClassifier rescales activations, and it is not a substitute for this point. Without a nonlinearity, stacking later layers would still be one linear function of the pixels.

### Demo

In [ ]:
import torch.nn as nn
import torch.optim as optim

X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_xor = torch.tensor([[0.], [1.], [1.], [0.]])
torch.manual_seed(0)

def fit_linear(steps=300):
    m = nn.Linear(2, 1)
    opt = optim.Adam(m.parameters(), lr=0.1)
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(steps):
        opt.zero_grad()
        loss = loss_fn(m(X).squeeze(-1), y_xor.squeeze(-1))
        loss.backward()
        opt.step()
    with torch.no_grad():
        pred = (torch.sigmoid(m(X).squeeze(-1)) > 0.5).float()
        acc = (pred == y_xor.squeeze(-1)).float().mean().item()
    return acc

acc_lin = fit_linear()
print('Linear-only XOR accuracy:', acc_lin)
assert acc_lin <= 0.75

In [ ]:
def fit_mlp(steps=2000):
    torch.manual_seed(1)
    m = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))
    opt = optim.Adam(m.parameters(), lr=0.2)
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(steps):
        opt.zero_grad()
        loss = loss_fn(m(X).squeeze(-1), y_xor.squeeze(-1))
        loss.backward()
        opt.step()
    with torch.no_grad():
        pred = (torch.sigmoid(m(X).squeeze(-1)) > 0.5).float()
        acc = (pred == y_xor.squeeze(-1)).float().mean().item()
    return acc

acc_mlp = fit_mlp()
print('MLP + ReLU XOR accuracy:', acc_mlp)
assert acc_mlp == 1.0

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
colors = ['C0' if v == 0 else 'C1' for v in y_xor.squeeze(-1).tolist()]
ax.scatter(X[:, 0], X[:, 1], c=colors, s=120)
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title('XOR: a line cannot separate; ReLU MLP can')
plt.tight_layout()
plt.show()

### Check

A stack of linear layers with no ReLU is still one linear map. What does that imply for XOR on the four corners? Answer first, then compare the demo above: linear accuracy should be at or below 0.75; the ReLU MLP should reach 1.0.

## 4. The train loop

### Theory

Training is the same story on every batch of crops. One **step**:

1. **Forward:** `logits = model(images)`
2. **Loss:** a scalar that is small when scores match targets (`cross_entropy_loss` today)
3. **Backward:** `loss.backward()` fills each parameter's `.grad` with \(\partial \mathrm{loss}/\partial \mathrm{param}\). You do not derive those by hand this week — optional appendix **00b** (`modules/00_nn_scratch`, `notebooks/00_neural_networks_and_autograd.ipynb`) does.
4. **Step:** `optimizer.step()` updates weights using those gradients. `optimizer.zero_grad()` **before** the next forward so gradients do not accumulate across batches.

A **batch** is one group of crops (here 16). An **epoch** is one full pass over the training split. Validation uses `evaluate()` — forward and metrics only, **no** optimizer step.

**Learning rate** is step size. Too small: loss crawls. Too large: updates overshoot; loss climbs or becomes NaN. The scaffold treats non-finite loss as a hard failure.

Optional deepener: [3Blue1Brown backpropagation](https://www.3blue1brown.com/lessons/backpropagation) — watch after you can narrate the four steps, not before.

In `train.py`, `train_epoch` loops batches, calls `zero_grad`, forward, loss, `backward`, `step`. `evaluate` sets `model.eval()`, runs forward only, and aggregates predictions for accuracy and recall.

**Why — those four steps, in that order.** The forward pass is the function you currently have. The loss turns a wrong class into one number the optimizer can see. Backward asks how that number would change if each weight moved a little. The step moves the weights. Skip the loss and there is no target. Skip backward and the step is a guess. Run them out of order and you apply yesterday's gradient to today's batch.

**Why — a batch, not one crop and not the whole log.** One crop is a noisy opinion about the weights. That pedestrian may be dark, partly hidden, or oddly framed. A batch averages a few of those opinions so the step is less twitchy. The full training set would be a calmer average and a much slower one: you would wait for every image before a single update. Sixteen crops is the compromise here. An epoch is one pass so every training crop gets a turn. The validation pass must not call `step`, or you would fit the split you use to judge the model.

**Why — zero the gradient.** `backward` adds into `.grad`. It does not replace it. Forget `zero_grad` and the next batch's gradient sits on top of the previous one. The step then chases a sum of old errors. That is not a larger learning rate. It is a corrupted direction.

**Why — learning rate is a distance.** The gradient says which way the loss decreases. The learning rate says how far to walk. Too small, and you barely leave the random initialization in the epochs you have. Too large, and you step past the valley: the loss climbs or becomes NaN. The scaffold treats a non-finite loss as a failed run, because the weights are no longer a function you can trust. The break demo is the same idea on a straight line, where the overshoot is visible without waiting on images.

### Demo

In [ ]:
def fit_line_lr(lr, steps=40):
    torch.manual_seed(0)
    x = torch.randn(64, 1)
    y = 2.0 * x
    m = nn.Linear(1, 1)
    opt = optim.SGD(m.parameters(), lr=lr)
    losses = []
    for _ in range(steps):
        opt.zero_grad()
        pred = m(x)
        loss = ((pred - y) ** 2).mean()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

losses_small = fit_line_lr(0.05)
losses_large = fit_line_lr(2.0)
print('final loss lr=0.05', losses_small[-1])
print('final loss lr=2.0 ', losses_large[-1])
assert losses_small[-1] < 0.05
assert losses_large[-1] > losses_small[-1] + 1.0

In [ ]:
cap = 1e4
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot([min(v, cap) for v in losses_small], label='lr=0.05')
ax.plot([min(v, cap) for v in losses_large], label='lr=2.0')
ax.set_yscale('log')
ax.set_xlabel('step')
ax.set_ylabel('MSE (capped for plot)')
ax.legend()
ax.set_title('Learning rate: stable vs divergent')
plt.tight_layout()
plt.show()

In [ ]:
print('Real train_loader: len=', len(train_loader), 'dataset=', len(train_loader.dataset))
print('train_epoch in train.py runs forward → loss → backward → step on each batch of crops.')

### Check

If `backward` runs twice on the same parameter and you **skip** `zero_grad`, what happens to `.grad`? The next cell runs two backward passes and compares gradient magnitude.

In [ ]:
tiny = nn.Linear(1, 1)
x1 = torch.tensor([[1.0]])
y1 = torch.tensor([[2.0]])
loss_fn = nn.MSELoss()
tiny.zero_grad()
loss1 = loss_fn(tiny(x1), y1)
loss1.backward()
g1 = tiny.weight.grad.abs().item()
loss2 = loss_fn(tiny(x1), y1)
loss2.backward()
g2 = tiny.weight.grad.abs().item()
print('grad norm after 1st backward', g1)
print('grad norm after 2nd backward without zero_grad', g2)
assert g2 > g1
print('If you forget zero_grad, gradients accumulate across batches.')

## 5. Why convolutions

### Theory

**Convolution** encodes two assumptions:

1. **Locality:** a 3×3 kernel looks at a neighborhood.
2. **Weight sharing:** the same kernel slides over the whole image, so a detector learned on the left also fires on the right.

Think in **channel stacks:** the stem's **32** output channels are 32 different 3×3 filters applied to RGB — 32 response maps at once. **Stage2** outputs **64** channels by convolving over those 32 maps, so each new channel mixes lower-level detectors; it is not 64 copies of one edge filter. After pooling, the **fc** layer reads the 64-dimensional channel vector as a feature summary for class scores.

Each output channel is one learned filter. **Stride 2** (with kernel 3, padding 1) halves height and width so the next layer sees a coarser grid. **Adaptive average pool** to 1×1 collapses space into one vector per channel so a linear layer can emit class scores.

**Scaffold `DrivingClassifier` blocks (do not redesign):**

- **stem:** `Conv2d(3,32,k=3,s=2,p=1,bias=False)` + BN + ReLU → `(B,32,32,32)` from `(B,3,64,64)`
- **stage2:** `Conv2d(32,64,...)` → `(B,64,16,16)`
- **pool:** `AdaptiveAvgPool2d(1,1)` + flatten → `(B,64)`
- **fc:** `Linear(64, num_classes)` → logits `(B, K)`

Parameter count stays modest because sharing repeats the same 3×3 weights across spatial locations instead of learning a separate weight per pixel.

**Why — not a dense layer on every pixel.** A linear layer from 64×64×3 into even one neuron needs a separate weight per pixel. Those weights do not know that a lane dash on the left is the same kind of mark as a lane dash on the right, so you relearn it at every location, and you need far more crops to pin the weights down. Neighboring pixels are the ones that form an edge. Pixels on opposite corners of a crop are often different objects. A dense map treats those cases as unrelated numbers.

**Why — a small filter that slides.** A 3×3 kernel asks a local question: does this neighborhood look like an edge, a blob, or a color change? Weight sharing asks that same question everywhere. That bias fits driving crops, where a person's outline is the same kind of pattern on the left of the patch and on the right. One kernel cannot recognize a whole person. It can only report a local pattern. Later layers combine the reports.

**Why — many channels, stacked.** One filter is one pattern. The stem's 32 channels are 32 different patterns on RGB, computed together. Each of stage2's 64 channels looks at that whole set of maps, so a higher channel can mean "vertical edge and a bright column together," not a second copy of the first edge. A person in a crop is not one 3×3 template. The class scores need the mix.

**Why — stride, then a pool that forgets location.** Stride 2 spends the same kind of filter on a coarser grid, so the next layer sees a wider piece of the crop without a giant kernel. Adaptive average pool to 1×1 then throws the grid away and keeps one number per channel. That matches the question "what is this crop," because the label has no position inside the patch. It is the wrong last step when the question is "where is the lane." We are not solving location this week. The pool is a choice of question.

### Demo

In [ ]:
import torch.nn.functional as F_conv

crop, _ = train_loader.dataset[0]
x1 = crop.unsqueeze(0)
kernel = torch.tensor([[-1., 0., 1.], [-1., 0., 1.], [-1., 0., 1.]])
k = torch.zeros(1, 1, 3, 3)
k[0, 0] = kernel
green = x1[:, 1:2, :, :]
resp = F_conv.conv2d(green, k, padding=1)
print('horizontal contrast response shape', tuple(resp.shape))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(crop.permute(1, 2, 0))
axes[0].set_title('Input crop')
axes[0].axis('off')
axes[1].imshow(resp[0, 0].detach(), cmap='gray')
axes[1].set_title('Green-channel conv response')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
cnn = DrivingClassifier(num_classes=4)
cnn.eval()
with torch.no_grad():
    s = cnn.stem(x1)
    t = cnn.stage2(s)
    emb = cnn.extract_features(x1)
    lg = cnn(x1)
print('input', tuple(x1.shape))
print('stem ', tuple(s.shape))
print('stage2', tuple(t.shape))
print('pooled embedding', tuple(emb.shape))
print('logits', tuple(lg.shape))
assert s.shape == (1, 32, 32, 32)
assert t.shape == (1, 64, 16, 16)
assert emb.shape == (1, 64)
assert lg.shape == (1, 4)

### Check

Kernel 3, stride 2, padding 1, input height 64. What is `H_out`, and why does **stem** therefore emit spatial size 32? The next cell evaluates the standard conv output formula.

In [ ]:
H, W, k, s, p = 64, 64, 3, 2, 1
H_out = (H + 2 * p - k) // s + 1
print('H_out formula gives', H_out)
assert H_out == 32
print('Stride 2 with k=3,p=1 halves 64→32 spatial size.')

## 6. Metrics that lie

### Theory

**Accuracy** = correct / all.

A **confusion matrix** has **rows = true class**, **columns = predicted class**. Diagonal entries are correct; off-diagonal cells are error types.

| | Predicted positive | Predicted negative |
|---|---|---|
| Actual positive | TP | FN |
| Actual negative | FP | TN |

Recall = TP / (TP + FN). Precision = TP / (TP + FP).

A four-class matrix is the same idea: one row and one column per class in `CLASS_NAMES`, with counts of true-vs-predicted pairs.

`per_class_recall` in the scaffold computes recall for each class id; `minority_recall` (your fill) is the same formula for one chosen class — pedestrian index 2 in the rubric.

**Why — accuracy under imbalance.** Accuracy counts every crop the same. If most crops are clear road, getting the rare pedestrian wrong still looks like a good score. The always-road rule is the extreme: no model, only the majority class, accuracy well above a coin flip on this set, pedestrian recall exactly zero. A driving log has the same skew. A metric that rewards the common class will certify a system that misses the uncommon one.

**Why — read the matrix, not only the diagonal.** One recall number says people were missed. It does not say what they were called. Rows are the true class and columns are the decision, so pedestrian-called-road is a different mistake from pedestrian-called-vehicle. Those two mistakes ask for different next labels. The diagonal is the part accuracy already told you.

**Why — recall for the pedestrian.** Recall asks: of the people who were there, how many did you find? Missing a pedestrian is the failure this module refuses to hide behind a single accuracy. Support sits in the denominator, which is why a class with a handful of people can look fine in accuracy and terrible in recall. `minority_recall` is that ratio, and you will write it.

**Why — precision is the other mistake.** Precision asks: of the crops you called pedestrian, how many were? A model that labels every crop pedestrian has perfect recall and useless precision. The car would brake for lane paint. Driving needs both. The trap you are graded on this week is the miss, because accuracy already hides it. Do not ignore precision when you read the matrix.

### Demo

In [ ]:
train_tgts = torch.tensor([train_loader.dataset[i][1].item() for i in range(len(train_loader.dataset))])
preds_road = torch.zeros(len(train_tgts), dtype=torch.long)
acc_road = accuracy(preds_road, train_tgts)
recalls_road = per_class_recall(preds_road, train_tgts, 4)
print('always-road accuracy', round(acc_road, 3))
print('per-class recalls', [round(r, 2) for r in recalls_road])
assert acc_road > 0.4
assert recalls_road[2] == 0.0

In [ ]:
from train import train_epoch, evaluate
import numpy as np

device = torch.device('cpu')
model = DrivingClassifier().to(device)
opt = optim.AdamW(model.parameters(), lr=1e-3)
for ep in range(4):
    train_epoch(model, train_loader, opt, cross_entropy_loss, device)
    loss, acc, recalls = evaluate(model, val_loader, cross_entropy_loss, device, 4)
    print(f'epoch {ep+1} val_acc={acc:.3f} recalls={[round(r, 2) for r in recalls]}')
ce_ped_recall = recalls[2]

In [ ]:
num_classes = 4
cm = np.zeros((num_classes, num_classes), dtype=np.int64)
model.eval()
with torch.no_grad():
    for imgs, tgts in val_loader:
        preds = model(imgs.to(device)).argmax(dim=-1).cpu().numpy()
        for p, t in zip(preds, tgts.numpy()):
            cm[t, p] += 1
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(num_classes), CLASS_NAMES, rotation=45, ha='right')
ax.set_yticks(range(num_classes), CLASS_NAMES)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
fig.colorbar(im, ax=ax, fraction=0.046)
plt.title('Validation confusion matrix (CE)')
plt.tight_layout()
plt.show()

### Check

Hand batch: preds `[2, 0, 2, 1]`, targets `[2, 2, 0, 2]`, class **2**. Compute support, TP, recall, and precision before you run the cell — then check against the asserts.

In [ ]:
preds_h = torch.tensor([2, 0, 2, 1])
tgts_h = torch.tensor([2, 2, 0, 2])
c = 2
support = (tgts_h == c).sum().item()
tp = ((preds_h == c) & (tgts_h == c)).sum().item()
recall = tp / support
pred_pos = (preds_h == c).sum().item()
precision = tp / pred_pos
batch_acc = (preds_h == tgts_h).float().mean().item()
print('support', support, 'TP', tp, 'recall', recall, 'precision', precision)
print('batch accuracy', batch_acc)
assert abs(recall - 1/3) < 1e-6
assert abs(precision - 0.5) < 1e-6
print('Accuracy on this tiny batch is not the number you would ship.')

## 7. The loss is the objective

### Theory

**Principle 2:** the loss is the definition of **good** that the optimizer sees.

**Class-weighted CE** multiplies each example by a weight that depends on its class (`weight` tensor of shape `(K,)` in PyTorch). We do not implement weighted CE this week — it sits between plain CE and focal loss.

**Focal loss** (Lin et al.) multiplies CE by a **focusing factor** that depends on how correct the model already is:

$$p_t = p_y, \qquad \mathrm{FL} = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

Derive the factor, do not only memorize the name:

- If \(\gamma = 0\), \((1-p_t)^0 = 1\), so FL matches CE when \(\alpha\) is absent.
- If the model is confident and **correct**, \(p_t \to 1\), \((1-p_t)^\gamma \to 0\) — the example stops dominating the gradient.
- If the model is unsure or **wrong**, \(p_t\) is small, \((1-p_t)^\gamma \approx 1\) — CE remains.
- \(\gamma = 2\) is the default in `TrainConfig.gamma`.
- \(\alpha_t\) is optional per-class weight, shape `(K,)`, applied as `alpha[targets]`; `None` means no extra class weight.

Focal loss does not replace inspection (Principle 3) or good labels — it changes which mistakes the step listens to.

**Why — the loss is what "good" means to the step.** The optimizer never sees your dashboard. It sees the scalar you differentiated. If that scalar is content with easy road, the weights move to serve easy road. Choosing plain cross-entropy, a class-weighted variant, or focal loss is choosing which crops may dominate the update.

**Why — a class weight is only half the problem.** A weight per class makes each pedestrian crop cost more than each road crop. That matches rarity. It does not look at whether the model already got the crop right. After the net is sure about most of the road, those easy road crops still each pay full \(-\log p_t\), and there are many of them. Rarity and difficulty are different. Weights fix rarity. They leave difficulty alone.

**Why — focal loss, and why the factor is \((1-p_t)^\gamma\).** Multiply \(-\log p_t\) by a term near 0 when \(p_t\) is near 1 and near 1 when \(p_t\) is small, so easy correct crops drop out of the gradient while hard or wrong crops keep their cross-entropy. \(\gamma = 0\) forces that factor to 1, which recovers plain cross-entropy and gives you a test. \(\gamma = 2\) is sharp enough that \(p_t = 0.99\) almost vanishes and \(p_t = 0.1\) barely shrinks. \(\alpha\) is an optional class weight on top, for when you want both rarity and difficulty. Focal loss does not create labels; it changes which mistakes the step listens to, and you still have to look at them.

### Demo

In [ ]:
import math

print('p_t | (1-p_t)^2 | -log(p_t) | focal (gamma=2)')
for p_t in [0.99, 0.90, 0.50, 0.10]:
    factor = (1 - p_t) ** 2
    ce = -math.log(p_t)
    fl = factor * ce
    print(f'{p_t:.2f} | {factor:.4f} | {ce:.3f} | {fl:.3f}')
assert (1 - 0.99) ** 2 < 0.001
assert (1 - 0.10) ** 2 > 0.5

### Check

From the focal demo table (\(\gamma=2\)): at \(p_t=0.99\), is \((1-p_t)^2\) below 0.001? At \(p_t=0.10\), is the factor above 0.5? The demo cell above printed the table and asserted both.

**FILL — `focal_loss`** in `modules/00_ml_gym/losses.py` only.

Signature: `focal_loss(logits, targets, gamma=2.0, alpha=None, reduction="mean")`.
Do **not** paste a solution into this notebook. Stable route hint: `log_softmax`, then gather the log prob of the true class. `gamma=0` and `alpha=None` must match `cross_entropy_loss`.

In [ ]:
try:
    torch.manual_seed(0)
    z = torch.randn(4, 4)
    t = torch.tensor([0, 1, 2, 3])
    fl0 = focal_loss(z, t, gamma=0.0)
    ce = cross_entropy_loss(z, t)
    print('gamma=0 allclose', torch.allclose(fl0, ce, atol=1e-5))
    assert torch.allclose(fl0, ce, atol=1e-5)
    z2 = torch.tensor([[8.0, 0.0, 0.0, 0.0]])
    t2 = torch.tensor([0])
    fl2 = focal_loss(z2, t2, gamma=2.0).item()
    ce2 = cross_entropy_loss(z2, t2).item()
    print('gamma=2 confident FL', fl2, 'CE', ce2, 'FL < CE', fl2 < ce2)
    assert fl2 < ce2
except NotImplementedError:
    print('STOP: implement focal_loss in modules/00_ml_gym/losses.py')

In [ ]:
try:
    focal_loss(torch.randn(2, 4), torch.tensor([0, 1]), gamma=2.0)
    torch.manual_seed(0)
    model_fl = DrivingClassifier().to(device)
    opt_fl = optim.AdamW(model_fl.parameters(), lr=1e-3)
    focal_fn = lambda lg, tg: focal_loss(lg, tg, gamma=2.0)
    for ep in range(4):
        train_epoch(model_fl, train_loader, opt_fl, focal_fn, device)
        _, acc_f, recalls_f = evaluate(model_fl, val_loader, focal_fn, device, 4)
        print(f'focal epoch {ep+1} val_acc={acc_f:.3f} recalls={[round(r, 2) for r in recalls_f]}')
    print(f'CE pedestrian recall (index 2): {ce_ped_recall:.3f}')
    print(f'Focal pedestrian recall (index 2): {recalls_f[2]:.3f}')
    print(f'Delta (focal - CE): {recalls_f[2] - ce_ped_recall:.3f}')
except NotImplementedError:
    print('Comparison waits until focal_loss fill is done')

## 8. Inspect and ship

### Theory

**Principle 3:** you cannot improve what you do not inspect.

Images are **NCHW** in `[0, 1]`. Return dict keys `path`, `n_errors`, `order`; write a figure even when `n_errors=0`.

After the gallery, you still run **break-it** (engineering hygiene), **pytest** (contracts), and `train.main` to export `artifacts/m00/<run_id>/metrics.json` — the same artifact path the course tracks for XP.

**Why — look at the mistakes, not another average.** Recall can rise while the errors that remain are the confident ones: the model is sure a person is road. Those are the crops that matter on a road, and the crops a labeling loop should queue. An average cannot show you the picture. A gallery can.

**Why — sort by confidence in the wrong class.** A low-confidence mistake is the model shrugging. A high probability on the wrong class is the model insisting. Insisting is the dangerous case, and it is the case you would send for another label. Sorting that way is the small form of hard-example mining. A fleet data engine is one large version of the same sort, not a different idea.

**Why — a rate you compute, a test, and a file.** `minority_recall` is the scalar the gallery does not replace: you still need the rate, and you write the ratio so the definition is yours. The gallery is from scratch because the point is the sort, not a library call. Pytest checks contracts you can get wrong while the pictures still look plausible. The break-it script repeats the always-road trap outside this notebook, so the metric lesson is not trapped in one session. The artifact is the run you can open tomorrow; opening the notebook is not the work.

**FILL — `minority_recall`** in `modules/00_ml_gym/metrics.py`.

Recall = TP / support for one class; **0** if support is 0. Hand example: preds `[2, 0, 2, 1]`, targets `[2, 2, 0, 2]`, class 2 → **1/3**.

In [ ]:
try:
    preds = torch.tensor([2, 0, 2, 1])
    tgts = torch.tensor([2, 2, 0, 2])
    mr = minority_recall(preds, tgts, minority_class=2)
    print('minority_recall', mr)
    assert abs(mr - 1/3) < 1e-6
except NotImplementedError:
    print('STOP: implement minority_recall in metrics.py')

**FROM SCRATCH — `build_error_gallery`** in `modules/00_ml_gym/error_gallery.py`.

Contract: sort mistakes by descending confidence of the wrong class. Do not implement it in this notebook.

In [ ]:
from error_gallery import build_error_gallery
from PIL import Image
import numpy as np

try:
    out = repo / 'artifacts' / 'm00' / 'nb_gallery.png'
    model.eval()
    imgs, tgts = next(iter(val_loader))
    with torch.no_grad():
        lg = model(imgs)
    res = build_error_gallery(imgs, tgts, lg, CLASS_NAMES, out)
    print(res)
    plt.imshow(np.array(Image.open(out)))
    plt.axis('off')
    plt.title('Error gallery')
    plt.show()
except NotImplementedError:
    print('STOP: implement build_error_gallery in error_gallery.py')

**Free response (Principle 2):** Which principle did focal loss apply? Why does \(\gamma=0\) match cross-entropy? Write 3–6 sentences before opening `solutions/00_ml_gym`.

_Write 3–6 sentences here._

**Free response (Principle 3):** The always-road classifier had high accuracy and zero pedestrian recall. Which principle says that metric was the wrong definition of good? Write 3–6 sentences.

_Write 3–6 sentences here._

### Demo

In [ ]:
from break_it_fix_it import main as break_demo
break_demo()

### Tests

```bash
python3 -m pytest modules/00_ml_gym -q
```

Scaffold tests in `test_scaffold.py` always run. `test_assignment_solutions.py` loads reference code from `solutions/00_ml_gym`. Student fill tests in `test_student_fills.py` skip until you implement the function.

In [ ]:
from train import main as train_main
metrics = train_main(TrainConfig(
    data_dir=repo / 'data' / 'm00_sample',
    epochs=2,
    loss_name='cross_entropy',
))
metrics_path = repo / 'artifacts' / 'm00' / metrics['run_id'] / 'metrics.json'
print('metrics path:', metrics_path)
print('minority_recall field:', metrics['minority_recall'])

**Come back cue**

Tomorrow: 25-min error-gallery review — implement build_error_gallery and re-run that cell.

Suggested slot: 25 minutes. 55 or 90 if you are also writing the principle cells.

In [ ]:
import sys
import json
from pathlib import Path

repo = Path.cwd()
if not (repo / 'modules' / '00_ml_gym').exists():
    repo = repo.parent
sys.path.insert(0, str(repo / 'modules'))
from common.progress import come_back_cue

print(come_back_cue('m00'))
progress_path = repo / 'artifacts' / 'progress.json'
if progress_path.is_file():
    with progress_path.open() as f:
        xp = json.load(f).get('xp', 0)
    print(f'XP so far: {xp}')